# 09 — Dəqiqləşdirmələr: payload sayı, cluster bootstrap CI, bypass sütunu, benchmark

1. **111 vs 114:** middleware testindəki 114 və 4b-dəki 111 fərqinin səbəbi; bir rəqəmə gətirmək.
2. **Cluster bootstrap CI:** semantikanın qorunması eyni payload-ın formaları müstəqil olmadığı üçün
   payload üzrə cluster bootstrap (1000 təkrar, seed = 42) ilə yenidən hesablanır.
3. **Bypass sütunu:** "MySQL-də işləyib detektordan keçən" bütün detektorlar üçün tam effektiv dəst üzrə yenidən.
4. **Benchmark:** 4 konfiqurasiya üçün p50/p95/p99 və req/s, orta ± std.

## 0. Mühit

In [1]:
import hashlib, sys, threading, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import joblib, numpy as np, pandas as pd, pymysql

SEED = 42
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import obfuscation as obf
RES_DIR, PROC_DIR, MODEL_DIR = ROOT / "results", ROOT / "data/processed", ROOT / "models"
V = pd.read_csv(PROC_DIR / "06_mysql_effectiveness.csv", keep_default_na=False)
VALID = [t for t in V.variant.unique() if t != "keyword_split_comment"]

## 1. 111 vs 114 — səbəb və vahid rəqəm
- **111** = orijinal forması effektiv olan payload-lar (semantika təhlilinin bazası — obfuskasiyanı orijinalla müqayisə etmək üçün orijinal effektiv olmalıdır).
- **114** = **hər hansı** etibarlı formada (orijinal və ya obfuskasiya) effektiv olan payload-lar.
- Fərq = 3 payload orijinal halında bizim sabit laborator kontekstlərində effektiv deyil (dırnaq balanssızlığı sintaksis xətası verir),
  amma obfuskasiya (əsasən `hex_strings` dırnaqları `0x…`-ə çevirir) bunu düzəldir və hücum işləyir.

**Vahid rəqəm:** ümumi "unikal effektiv payload" = **114** (ən azı bir formada işləyən). Semantikanın qorunması
yalnız orijinalı effektiv olan **111** payload üzrə hesablanır (müqayisə üçün orijinal baza lazımdır).

In [2]:
orig = V[V.variant == "original"]
orig_eff_ids = set(orig[orig.effective].payload_id)
any_eff_ids = set(V[V.effective & V.variant.isin(VALID)].payload_id)
extra = sorted(any_eff_ids - orig_eff_ids)
print(f"original effektiv: {len(orig_eff_ids)}")
print(f"hər hansı etibarlı formada effektiv (VAHİD RƏQƏM): {len(any_eff_ids)}")
print(f"yalnız obfuskasiya ilə effektiv olanlar: {len(extra)} → payload_id {extra}")
for pid in extra:
    techs = sorted(set(V[(V.payload_id == pid) & V.effective & V.variant.isin(VALID)].variant))
    print(f"  payload {pid}: {techs}  | {orig[orig.payload_id == pid].text.iloc[0][:60]!r}")
pd.Series({"unique_effective_any_form": len(any_eff_ids), "effective_original": len(orig_eff_ids),
           "effective_only_via_obfuscation": len(extra)}).to_frame("value").to_csv(RES_DIR / "09_payload_count_reconciliation.csv")

original effektiv: 111
hər hansı etibarlı formada effektiv (VAHİD RƏQƏM): 114
yalnız obfuskasiya ilə effektiv olanlar: 3 → payload_id [769, 799, 1202]
  payload 769: ['double_url_encode']  | "1'   )    ) "
  payload 799: ['combination', 'hex_strings']  | "1'||  (  select 'zbdv' where 1618  =  1618 or exp  (  ~  (  "
  payload 1202: ['hex_strings']  | "1'||  (  select 'ovlx' where 8767  =  8767 or sleep  (  5   "


## 2. Semantikanın qorunması — payload üzrə cluster bootstrap
İmza (key) sütunları 06-da saxlanmadığı üçün yalnız lazımi formaların imzalarını MySQL-dən yenidən çıxarırıq:
orijinalı effektiv olan 111 payload-un orijinal + etibarlı obfuskasiya formaları, 10 kontekst. Sanitizasiya formaları lazım deyil.

In [3]:
def load_env(path):
    return {k.strip(): v.strip() for k, v in (l.split("=", 1) for l in Path(path).read_text().splitlines()
                                              if "=" in l and not l.startswith("#"))}
ENV = load_env(ROOT / ".env")
DB = dict(host="localhost", user=ENV["MYSQL_EVAL_USER"], password=ENV["MYSQL_EVAL_PASSWORD"],
          database="sqli_ml_eval", autocommit=True, read_timeout=15, write_timeout=15, connect_timeout=5)
CONTEXT_SQL = {
    "string": "SELECT * FROM products WHERE name = '{p}'", "numeric": "SELECT * FROM products WHERE id = {p}",
    "string_p1": "SELECT * FROM products WHERE (name = '{p}')", "string_p2": "SELECT * FROM products WHERE ((name = '{p}'))",
    "dquote": 'SELECT * FROM products WHERE name = "{p}"', "dquote_p1": 'SELECT * FROM products WHERE (name = "{p}")',
    "dquote_p2": 'SELECT * FROM products WHERE ((name = "{p}"))', "numeric_p1": "SELECT * FROM products WHERE (id = {p})",
    "numeric_p2": "SELECT * FROM products WHERE ((id = {p}))", "like": "SELECT * FROM products WHERE name LIKE '%{p}%'",
}
CONTEXTS = {n: (lambda tpl: (lambda p: tpl.replace("{p}", p)))(tpl) for n, tpl in CONTEXT_SQL.items()}
BENIGN = {c: "0" if c.startswith("numeric") else "zzz_benign" for c in CONTEXTS}
SYNTAX_ERRORS, LEAK_ERRORS, TIMEOUT = {1064, 1149}, {1105, 1062, 1690}, 3024
_local = threading.local()
def conn():
    c = getattr(_local, "conn", None)
    if c is None or not c.open:
        c = pymysql.connect(**DB)
        with c.cursor() as cur: cur.execute("SET SESSION max_execution_time = 2000")
        _local.conn = c
    return c
def run(sql):
    c = conn(); t0 = time.perf_counter()
    try:
        with c.cursor() as cur:
            cur.execute(sql); rows = cur.fetchmany(1000)
        h = hashlib.sha1(repr(sorted(map(repr, rows))).encode()).hexdigest()[:12]; status, code = "ok", 0
    except pymysql.MySQLError as e:
        code = e.args[0] if e.args and isinstance(e.args[0], int) else -1
        status = ("syntax_error" if code in SYNTAX_ERRORS else "timeout" if code == TIMEOUT
                  else "conn_lost" if code in (2006, 2013) else "error")
        h = None
        if status == "conn_lost": _local.conn = None
    return {"status": status, "code": code, "rows_hash": h, "slow": time.perf_counter() - t0 >= 1.0}

sub = V[V.payload_id.isin(orig_eff_ids) & (V.variant.isin(VALID))][["payload_id", "variant", "text"]].copy()
texts = pd.unique(pd.concat([sub.text, pd.Series(["zzz_benign", "0"])]))
jobs = [(c, t) for c in CONTEXTS for t in texts]
print(f"{len(orig_eff_ids)} payload, {len(sub)} forma, {len(jobs)} icra")
t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as ex:
    sigs = list(ex.map(lambda j: run(CONTEXTS[j[0]](j[1])), jobs))
SIG = dict(zip(jobs, sigs))
print(f"MySQL: {time.perf_counter() - t0:.0f} s")

111 payload, 1137 forma, 10120 icra


MySQL: 141 s


In [4]:
def key(s): return (s["status"], s["code"] if s["status"] == "error" else 0, s["rows_hash"], s["slow"])
def eff(s, ctx):
    if s["status"] in ("syntax_error", "conn_lost"): return False
    if s["status"] == "timeout" or s["slow"]: return True
    if s["status"] == "error": return s["code"] in LEAK_ERRORS
    return key(s) != key(SIG[(ctx, BENIGN[ctx])])

# orijinal imzaları payload üzrə
orig_rows = sub[sub.variant == "original"].set_index("payload_id")
# hər obf forması üçün: orijinalın effektiv olduğu kontekstdə imza eyni qalıbmı → preserved
forms = sub[sub.variant != "original"].copy()
preserved = []
for r in forms.itertuples():
    ot = orig_rows.loc[r.payload_id, "text"]
    ok = False
    for ctx in CONTEXTS:
        if eff(SIG[(ctx, ot)], ctx) and key(SIG[(ctx, r.text)]) == key(SIG[(ctx, ot)]):
            ok = True; break
    preserved.append(ok)
forms["preserved"] = preserved
print(f"forma sayı: {len(forms)}, qorunan: {forms.preserved.sum()} "
      f"({forms.preserved.mean():.1%})")

# payload üzrə cluster bootstrap
from math import sqrt
pay_ids = forms.payload_id.unique()
grp = {pid: (int(forms[forms.payload_id == pid].preserved.sum()), int((forms.payload_id == pid).sum()))
       for pid in pay_ids}
rng = np.random.default_rng(SEED)
boot = []
for _ in range(1000):
    pick = rng.choice(pay_ids, size=len(pay_ids), replace=True)
    k = sum(grp[p][0] for p in pick); n = sum(grp[p][1] for p in pick)
    boot.append(k / n)
boot = np.array(boot)
point = forms.preserved.mean()
lo, hi = np.percentile(boot, [2.5, 97.5])
# müqayisə üçün sadə (kl+astersiz) Wilson
z = 1.959963984540054; k, n = int(forms.preserved.sum()), len(forms); p = k / n
d = 1 + z*z/n; c = (p + z*z/(2*n))/d; hw = z*sqrt(p*(1-p)/n + z*z/(4*n*n))/d
print(f"Semantika qorunur (point): {point:.1%}")
print(f"  payload cluster bootstrap 95% CI: [{lo:.1%}, {hi:.1%}]  (1000 təkrar, seed=42)")
print(f"  (müqayisə üçün forma-səviyyə Wilson: [{c-hw:.1%}, {c+hw:.1%}] — daha dar, çünki asılılığı nəzərə almır)")
pd.Series({"preserved_pct": round(point*100, 1), "cluster_ci_low": round(lo*100, 1), "cluster_ci_high": round(hi*100, 1),
           "n_forms": n, "n_payloads": len(pay_ids), "wilson_ci_low": round((c-hw)*100, 1),
           "wilson_ci_high": round((c+hw)*100, 1)}).to_frame("value").to_csv(RES_DIR / "09_semantics_cluster_bootstrap.csv")

forma sayı: 1026, qorunan: 881 (85.9%)


Semantika qorunur (point): 85.9%
  payload cluster bootstrap 95% CI: [84.6%, 86.9%]  (1000 təkrar, seed=42)
  (müqayisə üçün forma-səviyyə Wilson: [83.6%, 87.9%] — daha dar, çünki asılılığı nəzərə almır)


## 3. "MySQL-də işləyib detektordan keçən" — tam dəst, bütün detektorlar
Effektiv forma hovuzu = orijinal + etibarlı obfuskasiya, effektiv olanlar. Qara siyahı: `survives_*` (06-da hesablanıb).
LR variantları: model "təmiz" (0) deyirsə keçir. Hər detektor üçün forma sayı, unikal payload sayı və payload üzrə
cluster bootstrap 95% CI (seed = 42).

In [5]:
pool = V[V.variant.isin(["original"] + VALID) & V.effective].copy()
models = {"LR baseline": "lr_v2", "LR + normallaşdırma": "lr_norm_v2", "LR + norm. + augmentation": "lr_norm_aug_v2"}
for name, f in models.items():
    m = joblib.load(MODEL_DIR / f"{f}.joblib")
    pool[f"pass::{name}"] = m.predict(pool.text) == 0
pool["pass::Blacklist (Input)"] = pool.survives_input.astype(bool)
pool["pass::Blacklist (Search)"] = pool.survives_search.astype(bool)

DETS = ["Blacklist (Input)", "Blacklist (Search)", "LR baseline", "LR + normallaşdırma", "LR + norm. + augmentation"]
pids = pool.payload_id.values
uniq_pids = np.unique(pids)
rng = np.random.default_rng(SEED)
boot_idx = [rng.choice(uniq_pids, size=len(uniq_pids), replace=True) for _ in range(1000)]
by_pid = {pid: pool[pool.payload_id == pid] for pid in uniq_pids}

rows = []
for name in DETS:
    passed = pool[f"pass::{name}"]
    k, n = int(passed.sum()), len(pool)
    up = pool.loc[passed, "payload_id"].nunique()
    bvals = []
    for pick in boot_idx:
        kk = sum(int(by_pid[p][f"pass::{name}"].sum()) for p in pick)
        nn = sum(len(by_pid[p]) for p in pick)
        bvals.append(kk / nn)
    lo, hi = np.percentile(bvals, [2.5, 97.5])
    rows.append({"detector": name, "effective_forms": n, "passed_forms": k, "passed_unique_payloads": up,
                 "passed_pct": round(k / n * 100, 2), "cluster_ci_low": round(lo * 100, 2),
                 "cluster_ci_high": round(hi * 100, 2)})
bypass = pd.DataFrame(rows).set_index("detector")
bypass.to_csv(RES_DIR / "09_bypass_full_set.csv")
display(bypass)

,effective_forms,passed_forms,passed_unique_payloads,passed_pct,cluster_ci_low,cluster_ci_high
detector,,,,,,
Blacklist (Input),996,30,20,3.01,1.43,4.99
Blacklist (Search),996,123,22,12.35,7.88,17.74
LR baseline,996,46,23,4.62,2.92,6.43
LR + normallaşdırma,996,1,1,0.10,0.00,0.31
LR + norm. + augmentation,996,1,1,0.10,0.00,0.31


## 4. Benchmark yekun cədvəli

In [6]:
lat = pd.read_csv(RES_DIR / "middleware_latency.csv")
labels = {"none": "middleware yoxdur", "input": "sanitizeInput", "ml": "ML middleware", "both": "ML + sanitizeInput"}
tbl = pd.DataFrame({
    "konfiqurasiya": lat.config.map(labels),
    "p50 (ms)": lat.apply(lambda r: f"{r.p50_mean:.1f} ± {r.p50_std:.1f}", axis=1),
    "p95 (ms)": lat.apply(lambda r: f"{r.p95_mean:.1f} ± {r.p95_std:.1f}", axis=1),
    "p99 (ms)": lat.apply(lambda r: f"{r.p99_mean:.1f} ± {r.p99_std:.1f}", axis=1),
    "req/s": lat.apply(lambda r: f"{r.rps_mean:.0f} ± {r.rps_std:.0f}", axis=1),
}).set_index("konfiqurasiya")
display(tbl)
print("30s × 3 təkrar, 10 bağlantı, təmiz trafik (GET /products?search=wireless mouse)")
print(f"ML vs middleware-siz: median +{lat[lat.config=='ml'].p50_mean.iloc[0] - lat[lat.config=='none'].p50_mean.iloc[0]:.1f} ms, "
      f"req/s {lat[lat.config=='ml'].rps_mean.iloc[0]/lat[lat.config=='none'].rps_mean.iloc[0]-1:.0%}")

,p50 (ms),p95 (ms),p99 (ms),req/s
konfiqurasiya,,,,
middleware yoxdur,2.3 ± 0.6,3.7 ± 0.6,5.0 ± 1.0,3423 ± 281
sanitizeInput,3.0 ± 0.0,5.0 ± 1.0,6.7 ± 1.5,2908 ± 292
ML middleware,3.3 ± 0.6,5.7 ± 2.1,7.3 ± 3.2,2648 ± 403
ML + sanitizeInput,3.0 ± 0.0,4.7 ± 0.6,5.7 ± 0.6,2742 ± 166


30s × 3 təkrar, 10 bağlantı, təmiz trafik (GET /products?search=wireless mouse)
ML vs middleware-siz: median +1.0 ms, req/s -23%
